In [41]:
# 추천 시스템 실습에 사용할 기본 라이브러리 로드
from ast import literal_eval

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
# 소규모 MovieLens 평점 데이터 로드
ratings_path = "../../Data/ratings_small.csv"
data = pd.read_csv(ratings_path)

In [43]:
# 데이터 샘플 확인
data.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


### 1. user–item 행렬 만들기

- 현재 데이터는 `userId`, `movieId`가 행 단위로만 기록되어 있다.  
- 아이템 기반 협업 필터링(item-based collaborative filtering)을 적용하려면  
  **사용자 × 아이템(user × item)** 형태의 평점 행렬이 필요하다.

이제 `pivot_table`을 사용해 user–item 평점 테이블을 만든다.


In [44]:
# userId를 행, movieId를 열로 하는 user–item 평점 행렬 생성
data = data.pivot_table(values="rating", index="userId", columns="movieId")

In [45]:
# user–item 행렬 미리 보기
data.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# 행렬 크기(사용자 수, 아이템 수) 확인
data.shape

(671, 9066)

이제 사용자별로 각 영화에 준 평점을 한눈에 볼 수 있는  
user–item 테이블이 만들어졌다.

하지만 이 테이블에는 **영화 ID(`movieId`)만 있고, 영화 제목(`title`)이 없다.**  
다음 단계에서 영화 메타데이터를 불러와 제목을 붙이고, 다시 결합한다.

In [ ]:
# 평점 데이터와 영화 메타데이터 로드
ratings = pd.read_csv("../../Data/ratings_small.csv")
movies = pd.read_csv("../../Data/tmdb_5000_movies.csv")

In [48]:
# tmdb 데이터의 id 컬럼 이름을 ratings와 맞도록(movieId) 통일
movies.rename(columns={"id": "movieId"}, inplace=True)

In [49]:
# movieId 기준으로 평점 데이터와 영화 정보를 결합
ratings_movies = pd.merge(ratings, movies, on="movieId")

In [50]:
# 결합된 데이터 예시 한 줄 확인
ratings_movies.head(1)

,userId,movieId,rating,timestamp,budget,genres,homepage,keywords,original_language,original_title,...,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,1,2105,4.0,1260759139,11000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",NaN,"[{""id"": 3687, ""name"": ""graduation""}, {""id"": 61...",en,American Pie,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1999-07-09,235483004,95.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,There's nothing like your first piece.,American Pie,6.4,2296


In [51]:
# 결합 데이터의 크기(행, 열) 확인
ratings_movies.shape

(18571, 23)

In [52]:
# 사용자 × 영화 제목 기준 평점 행렬 생성 (결측치는 0으로 채움)
data = ratings_movies.pivot_table(
    values="rating",
    index="userId",
    columns="title",
).fillna(0)

In [53]:
# 사용자별 영화 평점 행렬 미리 보기
data.head()

title,10 Things I Hate About You,12 Angry Men,1408,15 Minutes,16 Blocks,"20,000 Leagues Under the Sea",2001: A Space Odyssey,2046,21 Grams,25th Hour,...,Willy Wonka & the Chocolate Factory,World Trade Center,X-Men Origins: Wolverine,Y Tu Mamá También,You Only Live Twice,"You, Me and Dupree",Young Frankenstein,Zodiac,eXistenZ,xXx
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
# 행렬 크기(사용자 수, 영화 수) 확인
data.shape

(670, 856)

이제 **사용자 × 영화 제목** 평점 행렬이 준비되었다.

하지만 아이템 기반 협업 필터링에서는  
행(row)이 **사용자**가 아니라 **아이템(영화)** 이어야 한다.  

코사인 유사도는 행 단위로 계산되므로,  
행을 영화 기준이 되도록 전치(transpose)해서 사용한다.


In [55]:
# 행과 열을 뒤집어 영화 × 사용자 행렬로 전치
data = data.transpose()
data.head(2)

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
12 Angry Men,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
# (영화 수, 사용자 수) 형태인지 다시 확인
data.shape

(856, 670)

전치 후에는 **행 = 영화, 열 = 사용자** 구조가 된다.

이제 각 영화 쌍의 평점 패턴을 비교해  
**영화 간 유사도**를 계산한다.

In [57]:
# 영화 간 코사인 유사도 행렬 계산
movie_sim = cosine_similarity(data, data)
print(movie_sim.shape)

(856, 856)


In [58]:
# 유사도 행렬을 DataFrame으로 변환 (행/열 모두 영화 제목)
movie_sim_df = pd.DataFrame(
    data=movie_sim,
    index=data.index,
    columns=data.index,
)

In [59]:
# 영화 유사도 행렬 일부 확인
movie_sim_df.head(3)

title,10 Things I Hate About You,12 Angry Men,1408,15 Minutes,16 Blocks,"20,000 Leagues Under the Sea",2001: A Space Odyssey,2046,21 Grams,25th Hour,...,Willy Wonka & the Chocolate Factory,World Trade Center,X-Men Origins: Wolverine,Y Tu Mamá También,You Only Live Twice,"You, Me and Dupree",Young Frankenstein,Zodiac,eXistenZ,xXx
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You,1.0,0.0,0.0,0.182153,0.0,0.022069,0.085323,0.0,0.0,0.10349,...,0.059856,0.0,0.161801,0.088076,0.0,0.0,0.097588,0.0,0.0,0.014121
12 Angry Men,0.0,1.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.00000,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000
1408,0.0,0.0,1.0,0.447214,0.0,0.173381,0.028245,0.0,0.0,0.00000,...,0.146955,0.0,0.148968,0.140265,0.0,0.0,0.191675,0.0,0.0,0.000000


이제 원하는 **기준 영화 하나를 고른 뒤**,  
그 영화와 가장 유사한 영화들을 **유사도 순으로 정렬**해 추천할 수 있다.

In [60]:
# 예시 1: 'X-Men Origins: Wolverine'와 유사한 영화 Top 9 (자기 자신 제외)
movie_sim_df["X-Men Origins: Wolverine"].sort_values(ascending=False)[1:10]

title
Romeo Must Die                        0.649625
The Wedding Planner                   0.631669
Dogtown and Z-Boys                    0.501189
An Unfinished Life                    0.485643
Conquest of the Planet of the Apes    0.474626
Reign Over Me                         0.458155
The Terminal                          0.445337
Young Frankenstein                    0.423840
Whale Rider                           0.394136
Name: X-Men Origins: Wolverine, dtype: float64

In [61]:
# 예시 2: 'Harry Potter and the Half-Blood Prince'와 유사한 영화 Top 9
movie_sim_df["Harry Potter and the Half-Blood Prince"].sort_values(ascending=False)[1:10]

title
Synecdoche, New York                      1.000000
The Blue Lagoon                           1.000000
Family Plot                               1.000000
Harry Potter and the Half-Blood Prince    1.000000
Liar Liar                                 1.000000
Once                                      1.000000
The Astronaut Farmer                      0.970143
Schindler's List                          0.724286
The Last King of Scotland                 0.707107
Name: Harry Potter and the Half-Blood Prince, dtype: float64

In [64]:
# (참고) 자기 자신을 포함한 Top 10 유사 영화
movie_sim_df["Harry Potter and the Half-Blood Prince"].sort_values(ascending=False)[:10]

title
Rendition                                 1.000000
Synecdoche, New York                      1.000000
The Blue Lagoon                           1.000000
Family Plot                               1.000000
Harry Potter and the Half-Blood Prince    1.000000
Liar Liar                                 1.000000
Once                                      1.000000
The Astronaut Farmer                      0.970143
Schindler's List                          0.724286
The Last King of Scotland                 0.707107
Name: Harry Potter and the Half-Blood Prince, dtype: float64

In [65]:
# 예시 3: 'King Kong'와 유사한 영화 Top 9
movie_sim_df["King Kong"].sort_values(ascending=False)[1:10]

title
Fantasia                                  0.648886
2046                                      0.648886
Synecdoche, New York                      0.486664
Once                                      0.486664
Liar Liar                                 0.486664
Family Plot                               0.486664
Harry Potter and the Half-Blood Prince    0.486664
The Blue Lagoon                           0.486664
Rendition                                 0.486664
Name: King Kong, dtype: float64

---
위 결과를 보면, 직관적으로 볼 때 추천 품질이 아주 만족스럽지는 않다.

주요 이유는 두 데이터셋  
(`ratings_small.csv`, `tmdb_5000_movies.csv`)의 영화 목록이 **완전히 일치하지 않기 때문**이다.  
평점 정보만 있거나, 메타데이터만 있는 영화가 많아서  
사용 가능한 영화가 제한된다.

이를 보완하기 위해, 다음부터는 MovieLens의  
**`ratings.csv` + `movies.csv`** 조합을 사용해  
좀 더 일관된 데이터로 동일한 방식의 추천 시스템을 구현한다.

In [66]:
# MovieLens 원본(대용량) 평점·영화 데이터 로드
rating_data = pd.read_csv("../../Data/ratings.csv")
movie_data = pd.read_csv("../../Data/movies.csv")

In [67]:
# 평점 데이터 샘플 확인
rating_data.head(2)

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179


In [68]:
# 영화 메타데이터 샘플 확인
movie_data.head(2)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy


두 파일은 각각

- 사용자–영화–평점 정보 (`ratings.csv`)
- 영화 메타데이터 (`movies.csv`)

를 담고 있다.

두 데이터는 공통 키 `movieId`를 공유하므로,  
이를 기준으로 결합(merge)하여 하나의 데이터프레임으로 만든다.  
그 후 `pivot_table`로 **사용자 × 영화** 평점 행렬을 만든다.

In [69]:
# 분석에 사용하지 않을 timestamp 컬럼 제거
rating_data.drop(columns="timestamp", inplace=True)
rating_data.head(2)

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0


In [74]:
# movieId 기준으로 평점 데이터와 영화 정보를 결합
user_movie_rating = pd.merge(rating_data, movie_data, on="movieId")

In [75]:
# 결합된 데이터 확인
user_movie_rating.head(2)

,userId,movieId,rating,title,genres
0,1,31,2.5,Dangerous Minds (1995),Drama
1,1,1029,3.0,Dumbo (1941),Animation|Children|Drama|Musical


In [76]:
# 아이템 기반 행렬: 영화(title)를 행, 사용자(userId)를 열로
movie_user_rating = user_movie_rating.pivot_table(
    values="rating",
    index="title",
    columns="userId",
)

# 사용자 기반 행렬: 사용자를 행, 영화를 열로
user_movie_rating = user_movie_rating.pivot_table(
    values="rating",
    index="userId",
    columns="title",
)

In [77]:
# 사용자 × 영화 평점 행렬 확인
user_movie_rating.head(5)

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931),İtirazım Var (2014)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
# 영화 × 사용자 평점 행렬 확인
movie_user_rating.head()

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
title,,,,,,,,,,,,,,,,,,,,,
"""Great Performances"" Cats (1998)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
$9.99 (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Neath the Arizona Skies (1934),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


지금 우리는 두 가지 형태의 평점 행렬을 갖고 있다.

- `movie_user_rating` : 행 = 영화, 열 = 사용자  
- `user_movie_rating` : 행 = 사용자, 열 = 영화  

먼저 **아이템 기반 협업 필터링(item-based CF)** 을 위해  
`movie_user_rating`에서 결측치(NaN)를 0으로 채워  
계산하기 쉽게 만든다.

In [79]:
# 아이템 기반 행렬의 결측치를 0으로 채움
movie_user_rating.fillna(0, inplace=True)
movie_user_rating.head(3)

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
title,,,,,,,,,,,,,,,,,,,,,
"""Great Performances"" Cats (1998)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
$9.99 (2008),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


지금 행렬 구조는 **행 = 영화, 열 = 사용자** 이다.

아이템 기반 협업 필터링의 해석은 대략 다음과 같다.

- “이 상품을 산 고객들은 저 상품도 함께 샀다.”  
- 영화 도메인에서는 “이 영화를 본(높게 평점 준) 사용자는 이런 영화도 좋아한다.”  

여기서는 영화 간 평점 패턴이 얼마나 비슷한지,  
즉 **코사인 유사도(cosine similarity)** 로  
영화 간 유사성을 계산한다.

In [80]:
# 영화 × 사용자 행렬을 이용해 영화 간 코사인 유사도 계산
item_based_collabor = cosine_similarity(movie_user_rating)
item_based_collabor

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.05821787, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.05821787, 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(9064, 9064))

In [81]:
# 유사도 행렬의 크기가 (영화 수, 영화 수)인지 확인
print(movie_user_rating.shape)
print(item_based_collabor.shape)

(9064, 671)
(9064, 9064)


In [82]:
# 유사도 배열을 DataFrame으로 변환 (행/열 이름을 영화 제목으로 설정)
item_based_collabor = pd.DataFrame(
    data=item_based_collabor,
    index=movie_user_rating.index,
    columns=movie_user_rating.index,
)

In [83]:
# 아이템 기반 유사도 행렬 일부 확인
item_based_collabor.head()

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931),İtirazım Var (2014)
title,,,,,,,,,,,,,,,,,,,,,
"""Great Performances"" Cats (1998)",1.000000,0.0,0.0,0.164399,0.020391,0.0,0.014046,0.000000,0.0,0.003166,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
$9.99 (2008),0.000000,1.0,0.0,0.000000,0.000000,0.0,0.000000,0.079474,0.0,0.156330,...,0.0,0.0,0.0,0.000000,0.0,0.013899,0.0,0.058218,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.000000,0.0,1.0,0.000000,0.000000,1.0,0.000000,0.217357,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
'Neath the Arizona Skies (1934),0.164399,0.0,0.0,1.000000,0.124035,0.0,0.085436,0.000000,0.0,0.019259,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
'Round Midnight (1986),0.020391,0.0,0.0,0.124035,1.000000,0.0,0.010597,0.143786,0.0,0.136163,...,0.0,0.0,0.0,0.121567,0.0,0.000000,0.0,0.000000,0.0,0.0


이제 영화(아이템) 간 유사도 행렬이 준비되었다.

마지막 단계는 **특정 영화 제목을 입력하면,  
그 영화와 가장 비슷한 영화들을 상위 N개 추천하는 함수**를 만드는 것이다.

In [84]:
def get_item_based_collabor(title: str, top_n: int = 6):
    """
    주어진 영화 제목과 코사인 유사도가 높은 영화들을
    상위 top_n개 반환한다. (자기 자신도 포함될 수 있음)
    """
    return item_based_collabor[title].sort_values(ascending=False)[:top_n]

In [85]:
# 예시: 'Godfather, The (1972)'와 유사한 영화 추천
get_item_based_collabor("Godfather, The (1972)")

title
Godfather, The (1972)                        1.000000
Godfather: Part II, The (1974)               0.773685
Goodfellas (1990)                            0.620349
One Flew Over the Cuckoo's Nest (1975)       0.568244
American Beauty (1999)                       0.557997
Star Wars: Episode IV - A New Hope (1977)    0.546750
Name: Godfather, The (1972), dtype: float64

---
위와 같이 특정 영화 제목을 넣으면  
해당 영화와 유사한 영화 목록을 쉽게 얻을 수 있다.

이 구조를 그대로 API나 서비스에 연결하면  
간단한 **아이템 기반 영화 추천 시스템**으로 확장할 수 있다.